# Stage 4b — STRONGER PdM brain (XGBoost on AI4I 2020)

Upgrades the Stage-4 MLP (PR-AUC 0.679, recall 0.61) using **XGBoost** — usually stronger on tabular data —
and reports a **recall-tuned threshold** so we catch more failures (gaps G-034 + G-033). Same clean,
**leakage-free** AI4I setup (leaky `TWF/HDF/PWF/OSF/RNF` dropped, stratified split). `Runtime ▸ Run all` →
downloads `pdm_xgb_brain.zip` — send it back. No GPU needed; no API key.

In [1]:
import numpy as np, pandas as pd, requests, zipfile, io, json, pickle, os
try:
    import xgboost as xgb
except Exception:
    import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','xgboost']); import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, precision_recall_curve, confusion_matrix
SEED=42; np.random.seed(SEED)
print('xgboost', xgb.__version__)

xgboost 3.2.0


In [2]:
url='https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip'
raw=pd.read_csv(zipfile.ZipFile(io.BytesIO(requests.get(url).content)).open('ai4i2020.csv'))
df=raw.copy()
df['Type_ord']=df['Type'].map({'L':0,'M':1,'H':2}).astype('float32')
df['temp_diff']=df['Process temperature [K]']-df['Air temperature [K]']
df['power_w']=df['Torque [Nm]']*df['Rotational speed [rpm]']*2*np.pi/60.0
FEATURES=['Type_ord','Air temperature [K]','Process temperature [K]','Rotational speed [rpm]','Torque [Nm]','Tool wear [min]','temp_diff','power_w']
X=df[FEATURES].values.astype('float32'); y=df['Machine failure'].values.astype('int')
print('pos rate', round(float(y.mean()),4), '(real ~3.4%) | dropped leaky TWF/HDF/PWF/OSF/RNF')

pos rate 0.0339 (real ~3.4%) | dropped leaky TWF/HDF/PWF/OSF/RNF


In [3]:
Xtr,Xtmp,ytr,ytmp=train_test_split(X,y,test_size=0.30,stratify=y,random_state=SEED)
Xva,Xte,yva,yte=train_test_split(Xtmp,ytmp,test_size=0.50,stratify=ytmp,random_state=SEED)
scaler=StandardScaler().fit(Xtr)  # XGBoost doesn't need it, but we keep one for a uniform inference path
spw=float((ytr==0).sum()/max((ytr==1).sum(),1))
clf=xgb.XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.9,
    colsample_bytree=0.9, scale_pos_weight=spw, eval_metric='aucpr', early_stopping_rounds=30, random_state=SEED)
clf.fit(Xtr,ytr, eval_set=[(Xva,yva)], verbose=False)
print('best_iteration', clf.best_iteration)

best_iteration 66


In [4]:
pte=clf.predict_proba(Xte)[:,1]; pva=clf.predict_proba(Xva)[:,1]
roc=roc_auc_score(yte,pte); prauc=average_precision_score(yte,pte)
prec,rec,thr=precision_recall_curve(yva,pva); f1=2*prec*rec/(prec+rec+1e-9)
THR_F1=float(thr[max(0,f1[:-1].argmax())])
# recall-tuned: lowest threshold giving val recall >= 0.80 (G-033)
rec_ok=np.where(rec[:-1]>=0.80)[0]; THR_RECALL=float(thr[rec_ok[-1]]) if len(rec_ok) else THR_F1
print(f'TEST ROC-AUC {roc:.3f} | PR-AUC {prauc:.3f} | baseline {yte.mean():.3f}')
for name,T in [('F1-opt',THR_F1),('recall>=0.80',THR_RECALL)]:
    pr=(pte>=T).astype(int); cm=confusion_matrix(yte,pr)
    print(f'\n[{name}] thr={T:.3f}  confusion [[TN,FP],[FN,TP]]={cm.tolist()}')
    print(classification_report(yte,pr,digits=3))

TEST ROC-AUC 0.971 | PR-AUC 0.847 | baseline 0.034

[F1-opt] thr=0.745  confusion [[TN,FP],[FN,TP]]=[[1438, 11], [10, 41]]
              precision    recall  f1-score   support

           0      0.993     0.992     0.993      1449
           1      0.788     0.804     0.796        51

    accuracy                          0.986      1500
   macro avg      0.891     0.898     0.894      1500
weighted avg      0.986     0.986     0.986      1500


[recall>=0.80] thr=0.779  confusion [[TN,FP],[FN,TP]]=[[1442, 7], [10, 41]]
              precision    recall  f1-score   support

           0      0.993     0.995     0.994      1449
           1      0.854     0.804     0.828        51

    accuracy                          0.989      1500
   macro avg      0.924     0.900     0.911      1500
weighted avg      0.988     0.989     0.989      1500



In [5]:
os.makedirs('brain',exist_ok=True)
clf.save_model('brain/pdm_failure_predictor_xgb.json')
pickle.dump(scaler,open('brain/scaler.pkl','wb'))
meta={'arch':'XGBoost','features':FEATURES,'type_encoding':{'L':0,'M':1,'H':2},
      'threshold_f1':round(THR_F1,4),'threshold_recall80':round(THR_RECALL,4),'task':'per-snapshot failure-risk (tabular)','seed':SEED,
      'scaler_mean':scaler.mean_.tolist(),'scaler_scale':scaler.scale_.tolist()}
json.dump(meta,open('brain/model_meta.json','w'),indent=2)
metrics={'roc_auc':round(float(roc),4),'pr_auc':round(float(prauc),4),'threshold_f1':round(THR_F1,4),
         'threshold_recall80':round(THR_RECALL,4),'test_positive_rate':round(float(yte.mean()),4),
         'n_test':int(len(yte)),'dataset':'AI4I 2020 (XGBoost, clean stratified split, no leakage)','model':'xgboost'}
json.dump(metrics,open('brain/metrics.json','w'),indent=2)
with zipfile.ZipFile('pdm_xgb_brain.zip','w',zipfile.ZIP_DEFLATED) as z:
    for fn in ['pdm_failure_predictor_xgb.json','scaler.pkl','model_meta.json','metrics.json']: z.write('brain/'+fn,fn)
print(json.dumps(metrics,indent=2))
try:
    from google.colab import files; files.download('pdm_xgb_brain.zip')
except Exception as e: print('grab pdm_xgb_brain.zip from the file browser', e)

{
  "roc_auc": 0.9713,
  "pr_auc": 0.847,
  "threshold_f1": 0.7451,
  "threshold_recall80": 0.779,
  "test_positive_rate": 0.034,
  "n_test": 1500,
  "dataset": "AI4I 2020 (XGBoost, clean stratified split, no leakage)",
  "model": "xgboost"
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
**Compare vs the MLP (Stage 4):** MLP was ROC-AUC 0.972 / PR-AUC 0.679 / recall 0.61. If XGBoost's PR-AUC is
higher and the recall-tuned threshold lifts recall toward 0.8 with acceptable precision, send `pdm_xgb_brain.zip`
and I'll wire it as the stronger brain (and pick the recall-tuned threshold for maintenance).